In [29]:
import numpy as np
import os
import pandas as pd
from sqlalchemy import create_engine, text
from glob import glob

# Database connections
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()

In [31]:
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dts_path = os.path.join(user_path, "OneDrive", "Downloads", "Datasets")
print(f"Datasets path: {dts_path}")
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data>
god_path = os.path.join(user_path, "OneDrive","Imports","santisoontarinka@gmail.com - Google Drive","Data")
#C:\Users\PC1\iCloudDrive\data
icd_path = os.path.join(user_path, "iCloudDrive", "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")

Datasets path: C:\Users\PC1\OneDrive\Downloads\Datasets


In [33]:
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Datasets path : {dts_path}") 
print(f"Data path : {dat_path}") 
print(f"Google Drive path : {god_path}")
print(f"iCloudDrive path : {icd_path}") 
print(f"Obsidian path : {osd_path}")

User path: C:\Users\PC1
Current path: C:\Users\PC1\OneDrive\A4\Data Cleansing
Base path: C:\Users\PC1\OneDrive\A4
Datasets path : C:\Users\PC1\OneDrive\Downloads\Datasets
Data path : C:\Users\PC1\OneDrive\A4\Data
Google Drive path : C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data
iCloudDrive path : C:\Users\PC1\iCloudDrive\Data
Obsidian path : C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data


In [35]:
sql = f"""
SELECT *
FROM stocks
ORDER BY name
"""
print(sql)
df_stocks_lt = pd.read_sql(sql, conlt)
print(df_stocks_lt.shape)
print(df_stocks_lt.dtypes)


SELECT *
FROM stocks
ORDER BY name

(229, 15)
id                int64
name             object
market           object
price           float64
max_price       float64
min_price       float64
pe              float64
pbv             float64
paid_up         float64
market_cap      float64
daily_volume    float64
beta            float64
ticker_id         int64
created_at       object
updated_at       object
dtype: object


In [37]:
sql = f"""
SELECT *
FROM stocks
ORDER BY name
"""
print(sql)
df_stocks_my = pd.read_sql(sql, conmy)
print(df_stocks_my.shape)
print(df_stocks_my.dtypes)


SELECT *
FROM stocks
ORDER BY name

(253, 15)
id                int64
name             object
market           object
price           float64
max_price       float64
min_price       float64
pe              float64
pbv             float64
paid_up         float64
market_cap      float64
daily_volume    float64
beta            float64
ticker_id         int64
created_at       object
updated_at       object
dtype: object


In [39]:
stocks_lt_set = set(df_stocks_lt['name'].to_list())
stocks_my_set = set(df_stocks_my['name'].to_list())
lt_my_dif_set = stocks_lt_set - stocks_my_set
lt_my_dif_set

{'AURA', 'BTG', 'MOSHI', 'STECON', 'TLI'}

In [41]:
# --- ADD MISSING STOCKS FROM portlt TO portmy (SQLite) ---
# This version handles database locks by using a fresh connection with a timeout.

print("\n" + "="*80)
print("SYNC STOCKS: portlt (SQLite) → portmy (SQLite)")
print("="*80)

# 1. Identify missing stocks in portmy
stocks_lt_set = set(df_stocks_lt['name'])
stocks_my_set = set(df_stocks_my['name'])
missing_stocks = stocks_lt_set - stocks_my_set

if not missing_stocks:
    print("✅ No missing stocks – portmy is already in sync with portlt.")
else:
    print(f"📌 Missing stocks in portmy: {len(missing_stocks)}")
    print(f"   {sorted(missing_stocks)}")

    # 2. Load ticker mappings from both databases
    df_tickers_lt = pd.read_sql("SELECT id, name FROM tickers", conlt)
    df_tickers_my = pd.read_sql("SELECT id, name FROM tickers", conmy)
    
    ticker_id_lt = dict(zip(df_tickers_lt['name'], df_tickers_lt['id']))
    ticker_id_my = dict(zip(df_tickers_my['name'], df_tickers_my['id']))
    
    # 3. Fetch missing stock records from portlt
    placeholders = ','.join(['?'] * len(missing_stocks))
    sql = f"""
        SELECT name, market, price, max_price, min_price, pe, pbv, 
               paid_up, market_cap, daily_volume, beta, ticker_id, 
               created_at, updated_at
        FROM stocks
        WHERE name IN ({placeholders})
    """
    df_missing = pd.read_sql(sql, conlt, params=tuple(missing_stocks))
    print(f"📥 Retrieved {len(df_missing)} stock records from portlt.")
    
    # 4. Map ticker_id from portlt to portmy
    ticker_name_for_id = {v: k for k, v in ticker_id_lt.items()}
    df_missing['ticker_name'] = df_missing['ticker_id'].map(ticker_name_for_id)
    df_missing['new_ticker_id'] = df_missing['ticker_name'].map(ticker_id_my)
    
    # Drop rows with missing tickers in portmy (should not happen if tickers are synced)
    before = len(df_missing)
    df_missing = df_missing.dropna(subset=['new_ticker_id'])
    if len(df_missing) < before:
        print(f"⚠️  Skipped {before - len(df_missing)} stocks because their ticker is missing in portmy.")
    
    # 5. Prepare DataFrame for insertion
    df_insert = df_missing.drop(columns=['ticker_id', 'ticker_name'], errors='ignore')
    df_insert.rename(columns={'new_ticker_id': 'ticker_id'}, inplace=True)
    
    # Ensure column order matches target table
    column_order = ['name', 'market', 'price', 'max_price', 'min_price', 'pe', 'pbv',
                    'paid_up', 'market_cap', 'daily_volume', 'beta', 'ticker_id',
                    'created_at', 'updated_at']
    df_insert = df_insert[column_order]
    
    # 6. Insert using a fresh connection with a timeout (to avoid "database is locked")
    #    Close the existing connection first, then create a new engine with timeout.
    conmy.close()  # release any lock held by the old connection
    
    # Create a new engine with a 30-second timeout
    engine_my_new = create_engine(
        "sqlite:///c:\\\\ruby\\\\portmy\\\\db\\\\development.sqlite3",
        connect_args={'timeout': 30}
    )
    
    try:
        with engine_my_new.connect() as conn:
            df_insert.to_sql('stocks', conn, if_exists='append', index=False)
        print(f"✅ Successfully added {len(df_insert)} stock records to portmy.")
    except Exception as e:
        if "UNIQUE constraint failed" in str(e) or "duplicate" in str(e).lower():
            print("⚠️  Some stocks already exist – using INSERT OR IGNORE to skip duplicates.")
            with engine_my_new.connect() as conn:
                for _, row in df_insert.iterrows():
                    conn.execute(
                        text("""
                            INSERT OR IGNORE INTO stocks 
                            (name, market, price, max_price, min_price, pe, pbv,
                             paid_up, market_cap, daily_volume, beta, ticker_id,
                             created_at, updated_at)
                            VALUES (:name, :market, :price, :max_price, :min_price, :pe, :pbv,
                                    :paid_up, :market_cap, :daily_volume, :beta, :ticker_id,
                                    :created_at, :updated_at)
                        """),
                        row.to_dict()
                    )
                conn.commit()
            print(f"✅ Inserted/ignored {len(df_insert)} records.")
        else:
            print(f"❌ Error: {e}")
            raise
    finally:
        engine_my_new.dispose()
    
    # 7. Recreate the original conmy connection for later cells (optional)
    engine = create_engine("sqlite:///c:\\\\ruby\\\\portmy\\\\db\\\\development.sqlite3")
    conmy = engine.connect()

# 8. Verification (using a fresh connection to avoid lock issues)
with engine_my_new.connect() as conn:
    df_stocks_my_after = pd.read_sql("SELECT name FROM stocks ORDER BY name", conn)
stocks_my_after_set = set(df_stocks_my_after['name'].tolist())
still_missing = missing_stocks - stocks_my_after_set

print("\n" + "-"*60)
print("VERIFICATION")
print("-"*60)
print(f"Stocks in portlt: {len(stocks_lt_set)}")
print(f"Stocks in portmy after sync: {len(stocks_my_after_set)}")
if not still_missing:
    print("✅ SUCCESS: All stocks from portlt are now present in portmy.")
else:
    print(f"⚠️  WARNING: Still missing {len(still_missing)} stocks: {sorted(still_missing)}")
print("="*80)


SYNC STOCKS: portlt (SQLite) → portmy (SQLite)
📌 Missing stocks in portmy: 5
   ['AURA', 'BTG', 'MOSHI', 'STECON', 'TLI']
📥 Retrieved 5 stock records from portlt.
⚠️  Skipped 5 stocks because their ticker is missing in portmy.
✅ Successfully added 0 stock records to portmy.

------------------------------------------------------------
VERIFICATION
------------------------------------------------------------
Stocks in portlt: 229
Stocks in portmy after sync: 253
⚠️  WARNING: Still missing 5 stocks: ['AURA', 'BTG', 'MOSHI', 'STECON', 'TLI']
